# Scope Drift Analysis

Run the scope drift model on citation networks. Configure the parameters below before running.

In [23]:
# ============================================================
# CONFIGURATION - Modify these settings before running
# ============================================================
import os

# Year range for network construction
os.environ["START_YEAR"] = "2025"
os.environ["END_YEAR"] = "2025"

# Network mode: "ego", "full", or "global"
# - "ego": Frontiers papers + direct citations
# - "full": Frontiers + related journals (default)
# - "global": ALL publications from ALL publishers (requires 64GB+ RAM)
os.environ["NETWORK_MODE"] = "full"

# Journals to analyze
# Option 1: Top N Frontiers journals by publication count
os.environ["TOP_N_JOURNALS"] = "5"

# Option 2: Specific journal IDs (uncomment to use)
# os.environ["JOURNAL_IDS"] = "910533066753,2972117368834,2379411881984,3315714752512,2405181685761"

# Clustering level for OOS detection: "macro", "meso", or "micro"
os.environ["JOURNAL_DRIFT_LEVEL"] = "meso"

# Leiden resolution - higher = more, smaller clusters (default meso: 0.000006)
os.environ["LEIDEN_RESOLUTION_MESO"] = "0.00005"
os.environ["MIN_COMMUNITY_SIZE"] = "200"

# ---- Edge Weight Tuning (for better weight distribution) ----
# Lower tau = older citations weigh much less (increases variance)
os.environ["TEMPORAL_DECAY_TAU"] = "2.0"  # default 5.0, try 2.0-3.0

# Lower = more penalty for same-journal citations (breaks up journal cliques)
os.environ["SELF_CITE_JOURNAL_WEIGHT"] = "0.2"  # default 0.5, try 0.2-0.3

# Higher = only strong bibliographic coupling edges (clearer signal)
os.environ["BC_MIN_SHARED_REFS"] = "5"  # default 3, try 5-8

# Weight transform to spread distribution: "none", "log", "sqrt", or "power:0.3"
os.environ["WEIGHT_TRANSFORM"] = "log"  # log works well for spreading weights

# Remove weak edges below this threshold (creates clearer community boundaries)
os.environ["EDGE_WEIGHT_THRESHOLD"] = "0.1"  # try 0.05-0.2

# Contrast: amplify differences (>1 = strong edges stronger, weak edges weaker)
os.environ["WEIGHT_CONTRAST"] = "2.0"  # try 1.5-3.0

print("Configuration set:")
print(f"  Year range: {os.environ['START_YEAR']} - {os.environ['END_YEAR']}")
print(f"  Network mode: {os.environ['NETWORK_MODE']}")
print(f"  Top N journals: {os.environ['TOP_N_JOURNALS']}")

Configuration set:
  Year range: 2025 - 2025
  Network mode: full
  Top N journals: 5


In [24]:
os.getcwd()

'c:\\Users\\sophie.wilson\\Documents\\scope-drift-model'

In [ ]:
# ============================================================
# RUN SCOPE DRIFT ANALYSIS
# ============================================================
import importlib.util
from pathlib import Path

# Get the directory where this notebook is located (works on Windows & Linux)
notebook_dir = Path().absolute()
module_path = notebook_dir / "src" / "scope_drift.py"

print(f"Loading module from: {module_path}")

# Load scope_drift module
spec = importlib.util.spec_from_file_location("scope_drift", module_path)
scope_drift = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scope_drift)

# Run the analysis
results = scope_drift.main()

2026-06-04 10:53:08,692 [INFO] ============================================================
2026-06-04 10:53:08,693 [INFO] SCOPE DRIFT — Global Citation Network Analysis
2026-06-04 10:53:08,694 [INFO]   Mode: FULL, Years: 2025-2025
2026-06-04 10:53:08,695 [INFO] ============================================================
2026-06-04 10:53:08,695 [INFO] Getting top 5 Frontiers journals by publication count...


Loading module from: c:\Users\sophie.wilson\Documents\scope-drift-model\src\scope_drift.py
Loading module from: c:\Users\sophie.wilson\Documents\scope-drift-model\src\create_html_output.py


c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-06-04 10:53:12,520 [INFO] Top Frontiers journals:
    JournalId                DisplayName  pubs
3315714752512    Frontiers in Immunology  7031
2379411881984 Frontiers in Public Health  5426
2826088480769      Frontiers in Medicine  4837
2972117368834      Frontiers in Oncology  4576
2405181685761    Frontiers in Psychology  4267
2026-06-04 10:53:12,521 [INFO] Fetching Frontiers publication IDs...
c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-06-04 10:53:15,413 [INFO] Frontiers publications: 26,137
2026-06-04 10:53:15,415 [INFO] Building EXTENDED citation network for 2025-2

In [ ]:
# ============================================================
# LOAD OUTPUT FILES (if viewing results later)
# ============================================================
import json
from pathlib import Path

output_dir = Path("output")

# Load the JSON results
json_file = output_dir / "scope_global_network.json"
if json_file.exists():
    with open(json_file) as f:
        saved_results = json.load(f)
    print(f"Loaded results from {json_file}")
    print(f"  Journals analyzed: {len(saved_results['journals'])}")
else:
    print(f"No results file found at {json_file}")
    print("Run the analysis first.")

Loaded results from output\scope_global_network.json
  Journals analyzed: 5


In [ ]:
# ============================================================
# VIEW RESULTS SUMMARY
# ============================================================

if saved_results:
    print("=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    meta = saved_results.get("meta", {})
    print(
        f"\nClustering: {saved_results.get('clustering_method', 'leiden')} at {meta.get('primary_cluster_level', 'meso')} level"
    )
    print(f"Year range: {meta.get('year_range', ['?', '?'])}")

    # ---- Largest Communities ----
    print("\n" + "-" * 70)
    print("LARGEST COMMUNITIES")
    print("-" * 70)
    communities = saved_results.get("communities", [])
    for comm in communities[:15]:  # Top 15 largest
        print(f"  [{comm['id']:3d}] {comm['label']:<45} Size: {comm['size']:,}")

    # ---- Journal Summary with Top Communities ----
    print("\n" + "-" * 70)
    print("JOURNAL SUMMARY & TOP COMMUNITIES")
    print("-" * 70)

    for j in saved_results["journals"]:
        print(f"\n{j['name'].upper()}")
        print(
            f"  Articles: {j['articles']:,}  |  OOS: {j['out_of_scope_pct']:.1f}%  |  Primary clusters: {j['n_primary_clusters']}"
        )
        print(f"  Top communities (in-scope marked with *):")
        for tc in j["top_communities"][:8]:
            marker = "*" if tc["is_primary"] else " "
            print(
                f"    {marker} [{tc['comm_id']:3d}] {tc['label']:<40} {tc['papers_in_comm']:,} papers ({tc['share_of_journal']:.1f}%)"
            )

RESULTS SUMMARY

Clustering: leiden at meso level
Year range: [2025, 2025]

----------------------------------------------------------------------
LARGEST COMMUNITIES
----------------------------------------------------------------------
  [  0] Immunology                                    Size: 20,718
  [  1] Education & Educational Research              Size: 7,888
  [  2] Urology & Nephrology - General                Size: 4,443
  [  3] Immunology                                    Size: 2,707
  [  4] Virology - General                            Size: 2,304
  [  5] Immunology                                    Size: 1,628
  [  6] Immunology                                    Size: 1,349
  [  7] Breast Cancer Scanning                        Size: 1,299
  [  8] Liver & Colon Cancer                          Size: 1,283
  [  9] Healthcare Policy                             Size: 1,136
  [ 10] Assisted Ventilation                          Size: 1,124
  [ 11] Molecular & Cell Biology - 

In [ ]:
# ============================================================
# PARAMETER TUNING - Test different thresholds automatically
# ============================================================
# Run this cell to test multiple parameter combinations and find the best clustering

import subprocess
import sys

# Quick mode tests 8 combinations, full mode tests 144
mode = "--quick"  # Change to "--full" for exhaustive search

print("Running parameter tuning...")
print(
    "This will test multiple configurations and report which gives the best clustering.\n"
)

result = subprocess.run(
    [sys.executable, "src/tune_clustering.py", mode],
    cwd=str(Path().absolute()),
    capture_output=False,
)

print("\nDone! Check output/tuning_results.json for full results.")

Loaded results from output\scope_global_network.json
  Journals analyzed: 5
